In [10]:
import pandas as pd
import numpy as np
import altair as alt

# Habilitar el entorno visual
alt.renderers.enable('colab')

# Cargar la matriz de datos limpia
df = pd.read_csv('06_Recuento_noticias.csv', sep=';')

# Ajustes del espacio
df_mosaic = df.copy()
df_mosaic = df_mosaic.sort_values("Apariciones", ascending=False).reset_index(drop=True)
total_apariciones = df_mosaic["Apariciones"].sum()

# Cálculos de geometría para el Muro
df_mosaic["prop"] = df_mosaic["Apariciones"] / total_apariciones
df_mosaic["x0"] = df_mosaic["prop"].cumsum() - df_mosaic["prop"]
df_mosaic["x1"] = df_mosaic["prop"].cumsum()
df_mosaic["xmid"] = (df_mosaic["x0"] + df_mosaic["x1"]) / 2

# Mostrar nombre
df_mosaic["etiqueta_grafico"] = np.where(df_mosaic["Apariciones"] >= 4, df_mosaic["Nombre común"], "")

# Colores
rce_colors = ["#F57C00", "#4CAF50"]
rce_domain = ["CR", "EN"]

# Bloque visual
blocks = alt.Chart(df_mosaic).mark_rect(stroke="white", strokeWidth=1).encode(
    x=alt.X("x0:Q", title="Distribución del Espacio Mediático", axis=alt.Axis(format="%")),
    x2="x1:Q",
    # Altura
    y=alt.value(50),
    y2=alt.value(350),
    color=alt.Color("RCE:N", scale=alt.Scale(domain=rce_domain, range=rce_colors), title="Categoría de Riesgo"),

    # Ficha Técnica interactiva
    tooltip=[
        alt.Tooltip("Nombre común:N", title="Nombre Común"),
        alt.Tooltip("Nombre científico:N", title="Nombre Científico"),
        alt.Tooltip("RCE:N", title="Estado de Conservación"),
        alt.Tooltip("Ficha técnica:N", title="Ficha Técnica Descriptiva")
    ]
)

# Textos
labels = alt.Chart(df_mosaic).mark_text(
    align="center",
    baseline="middle",
    color="white",
    fontSize=12,
    fontWeight="bold",
    angle=270
).encode(
    x=alt.X("xmid:Q"),
    # Anclaje
    y=alt.value(200),
    text="etiqueta_grafico:N"
)

# Ensamblaje y aplicación de tipografía
mosaico_final = (blocks + labels).properties(
    width=850,
    height=400,
    title={
        "text": "LA ASFIXIA DEL ESPACIO MEDIÁTICO",
        "subtitle": ["Mientras algunas especies brillan en su protagonismo, otras son invisibles para los medios."],
        "color": "#4A148C",
        "fontSize": 22,
        "subtitleColor": "#D10074"
    }
).configure_view(strokeWidth=0)

# Exportación para la webstory
mosaico_final.save('mosaico_invisibilidad.html')
mosaico_final

alt.LayerChart(...)